# Prepare Environment
<hr>

In [1]:
import os

# Remove the exported function definitions that cause the noise
for key in list(os.environ.keys()):
    if key.startswith("BASH_FUNC_"):
        del os.environ[key]

if os.path.exists('.bedrock_agentcore.yaml'):
    os.remove('.bedrock_agentcore.yaml')

In [2]:
!pip install -r requirements.txt --quiet

#### Create Demo Role

In [3]:
import boto3
import json

ACCOUNT_ID = "869795389844"
ROLE_NAME = "AgentCoreDemoRole"

iam = boto3.client("iam")

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": [
                    "sagemaker.amazonaws.com",
                    "ec2.amazonaws.com",
                    "lambda.amazonaws.com",
                    "apigateway.amazonaws.com",
                    "bedrock-agentcore.amazonaws.com",
                ]
            },
            "Action": "sts:AssumeRole",
        },
    ],
}

permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "*",
            "Resource": "*",
        }
    ],
}

try:
    response = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Demo role for AgentCore notebook testing - wide open permissions",
    )
    role_arn = response["Role"]["Arn"]
    print(f"Created role: {role_arn}")
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
    iam.update_assume_role_policy(
        RoleName=ROLE_NAME,
        PolicyDocument=json.dumps(trust_policy),
    )
    print(f"Role already exists, updated trust policy: {role_arn}")

iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="DemoFullAccess",
    PolicyDocument=json.dumps(permissions_policy),
)

print(f"Attached permissions policy")
print(f"\nRole ARN: {role_arn}")

Role already exists, updated trust policy: arn:aws:iam::869795389844:role/AgentCoreDemoRole
Attached permissions policy

Role ARN: arn:aws:iam::869795389844:role/AgentCoreDemoRole


# Standalone Agent on Local Environment (Jupyter Notebook)
<hr>

In [4]:
import json
import argparse

from strands import Agent
from strands.models import BedrockModel

model_id = "global.anthropic.claude-opus-4-6-v1"
model = BedrockModel(
    model_id=model_id,
)

agent = Agent(
    model=model,
    system_prompt="You're a helpful assistant."
)

In [5]:
response = agent("Top 5 biggest countries by size ?")

Here are the **top 5 biggest countries by land area**:

1. **Russia** — ~17.1 million km²
2. **Canada** — ~10.0 million km²
3. **United States** — ~9.8 million km²
4. **China** — ~9.6 million km²
5. **Brazil** — ~8.5 million km²

Russia is by far the largest, spanning both Europe and Asia and covering more than **11%** of the world's total landmass.

In [6]:
response = agent("What is the weather in Athens now ?")

I'm sorry, but I don't have the ability to access real-time data, including current weather conditions. I can't browse the internet or retrieve live information.

To check the current weather in Athens, I'd recommend:

- **Google**: Simply search "weather in Athens"
- **Weather apps**: Such as **AccuWeather**, **Weather.com**, or **The Weather Channel**
- **Smartphone**: Check the built-in weather app on your phone

That said, if you'd like general information about Athens' climate for a particular time of year, I'd be happy to help with that!

#### Adding a tool

In [7]:
from strands_tools.tavily import tavily_search

os.environ["TAVILY_API_KEY"] = "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"
#TODO : Add DUCKDUODO scenario

agent = Agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You're a helpful assistant."
)

In [8]:
response = agent("What is the weather in Athens now ?")


Tool #1: tavily_search


╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: current weather in Athens now                                                                            │
│                                                                                                                 │
│ Answer: Athens has clear skies, a temperature of 14°C, and light winds. The forecast is for mild weather with   │
│ occasional rain.                                                                                                │
│                                                                                                                 │
│ Results: 5 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] Weather in Athens                                                                                           │
│ URL: https://www.weatherapi.com/                                                                                │
│ Score: 0.9593857                                                                                                │
│ Content: {'location': {'name': 'Athens', 'region': 'Attica', 'country': 'Greece', 'lat': 37.9795, 'lon':        │
│ 23.7162, 'tz_id': 'Europe/Athens', 'localtime_epoch': 1777057914, 'localtime': '2026-04-24 22:11'}, 'current':  │
│ {'last_updated_epoch': 1777057200, 'last_updated': '2026-04-24 22:00', 'temp_c': 14.4, 'temp_f': 57.9,          │
│ 'is_day': 0, 'condition': {'text': 'Clear', 'icon': '//cdn.weatherapi.com/weather/64x64/night/113.png', 'code': │
│ 1000}, 'wind_mph': 6.7, 'wind_kph': 10.8, 'wind_degree': 12, 'wind_dir': 'NNE', 'pressure_mb': 1013.0,          │
│ 'pressure_in': 29.91, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 63, 'cloud': 25, 'feelslike_c': 13.8,     │
│ 'feelslike_f': 56.8, 'windchill_c': 12.4, 'windchill_f': 54.4, 'heatindex_c': 13.3, 'heatindex_f': 55.9,        │
│ 'dewpoint_c': 2.8, 'dewpoint_f': 37.1, 'vis_km': 10.0, 'vis_miles': 6.0, 'uv': 0.0, 'gust_mph': 11.9,           │
│ 'gust_kph': 19.2}}                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Weather for Athens, Greece - Time and Date                                                                  │
│ URL: https://www.timeanddate.com/weather/greece/athens                                                          │
│ Score: 0.9006087                                                                                                │
│ Content: Weather in Athens, Greece ; Apr 24, 2026 at 3:00 am · N/A · 29.95 "Hg · 64% · 40 °F                    │
│                                                                                                                 │
│                                                                                                                 │
│ [3] Athens, Attica, Greece Monthly Weather - AccuWeather                                                        │
│ URL: https://www.accuweather.com/en/gr/athens/182536/april-weather/182536                                       │
│ Score: 0.82492816                                                                                               │
│ Content: Athens's April 2026 forecast shows daily high temperatures ranging from 60° to 75°, with overnight     │
│ lows between 47° and 58°. The average high for April is                                                         │
│                                                       

Here's the current weather in **Athens, Greece** 🇬🇷:

| Detail | Value |
|---|---|
| **Condition** | ☀️ Clear skies |
| **Temperature** | 14.4°C (57.9°F) |
| **Feels Like** | 13.8°C (56.8°F) |
| **Humidity** | 63% |
| **Wind** | 10.8 km/h (6.7 mph) from NNE |
| **Wind Gusts** | 19.2 km/h (11.9 mph) |
| **Pressure** | 1013 mb |
| **Visibility** | 10 km (6 miles) |
| **Cloud Cover** | 25% |
| **UV Index** | 0 (nighttime) |

It's currently nighttime in Athens (around 10:11 PM local time) with clear skies and mild temperatures. A light jacket would be a good idea if you're heading out! 🌙

#### Adding a custom tool

In [9]:
from strands import tool

# Weather tool
@tool
def weather(city: str):
    """Get weather information.

    Args:
        city: City for which weather will be returned

    Returns:
        Weather of provided city as a string.
    """
    print(city)
    if city.lower() == "athens":
        weather = "very sunny"
    else:
        weather = "sunny"
    return weather

In [10]:
agent = Agent(
    model=model,
    tools=[weather, tavily_search],
    system_prompt="You're a helpful assistant."
)

response = agent("What is the weather in Athens now ?")


Tool #1: weather
Athens
The weather in Athens right now is **very sunny**! ☀️ A great day to be out and about in the city. Is there anything else you'd like to know?

In [11]:
response = agent("What is the weather in Athens now ? Cross check with information from the web")



Let me get you the weather from both sources simultaneously!
Tool #2: weather

Tool #3: tavily_search
Athens


╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: current weather in Athens now                                                                            │
│                                                                                                                 │
│ Answer: Athens has a clear night with a temperature of 14.4°C (57.9°F) and a light breeze from the northeast.   │
│ The humidity is 63%, and there is no precipitation. The forecast is for clear conditions.                       │
│                                                                                                                 │
│ Results: 5 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] Weather in Athens                                                                                           │
│ URL: https://www.weatherapi.com/                                                                                │
│ Score: 0.9998975                                                                                                │
│ Content: {'location': {'name': 'Athens', 'region': 'Attica', 'country': 'Greece', 'lat': 37.9795, 'lon':        │
│ 23.7162, 'tz_id': 'Europe/Athens', 'localtime_epoch': 1777057914, 'localtime': '2026-04-24 22:11'}, 'current':  │
│ {'last_updated_epoch': 1777057200, 'last_updated': '2026-04-24 22:00', 'temp_c': 14.4, 'temp_f': 57.9,          │
│ 'is_day': 0, 'condition': {'text': 'Clear', 'icon': '//cdn.weatherapi.com/weather/64x64/night/113.png', 'code': │
│ 1000}, 'wind_mph': 6.7, 'wind_kph': 10.8, 'wind_degree': 12, 'wind_dir': 'NNE', 'pressure_mb': 1013.0,          │
│ 'pressure_in': 29.91, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 63, 'cloud': 25, 'feelslike_c': 13.8,     │
│ 'feelslike_f': 56.8, 'windchill_c': 12.4, 'windchill_f': 54.4, 'heatindex_c': 13.3, 'heatindex_f': 55.9,        │
│ 'dewpoint_c': 2.8, 'dewpoint_f': 37.1, 'vis_km': 10.0, 'vis_miles': 6.0, 'uv': 0.0, 'gust_mph': 11.9,           │
│ 'gust_kph': 19.2}}                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Athens weather in April 2026 - Weather25.com                                                                │
│ URL: https://www.weather25.com/europe/greece/attiki/athens?page=month&month=April                               │
│ Score: 0.99985456                                                                                               │
│ Content: The temperatures in Athens in April are usually low and can range between 55°F and 66°F. You can       │
│ expect about 3 to 8 days of rain in Athens during the month                                                     │
│                                                                                                                 │
│                                                                                                                 │
│ [3] Athens, Attica, Greece Monthly Weather - AccuWeather                                                        │
│ URL: https://www.accuweather.com/en/gr/athens/182536/april-weather/182536                                       │
│ Score: 0.99980444                                                                                               │
│ Content: Athens's April 2026 forecast shows daily high temperatures ranging from 60° to 75°, with overnight     │
│ lows between 47° and 58°. The average high for April i

Here's a comparison of the weather in Athens from both sources:

| Source | Weather Details |
|--------|----------------|
| **Weather Tool** | Very sunny ☀️ |
| **Web Search** | **Clear** skies, 14.4°C (57.9°F), humidity at 63%, light breeze from the NNE at 10.8 km/h, no precipitation |

**Cross-check summary:**
- Both sources generally agree — the weather tool reports "very sunny" conditions, while the web data (from WeatherAPI, as of **April 24, 2026 at 10:00 PM local time** in Athens) reports **clear skies** with no precipitation. Since it's nighttime in Athens right now, "clear" is the more accurate description, but both indicate fair, dry weather with no clouds or rain.
- The temperature is a mild **14.4°C (57.9°F)**, feels like **13.8°C**, with good visibility of 10 km.

Overall, both sources confirm **pleasant, clear weather** in Athens right now! 🌙

# Standalone Agent on AgentCore
<hr>

In [12]:
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime


boto_session = Session()
region = boto_session.region_name

<img src="images/agent_core_configure_and_launch.png" width="1100">

In [13]:
agent_runtime = Runtime()

agent_runtime.configure(
    agent_name="agent",
    entrypoint="01_agent.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region
)

Entrypoint parsed: file=C:\allwyn\01_agent.py, bedrock_agentcore_name=01_agent
Configuring BedrockAgentCore agent: agent


⚠️  ℹ️  No container engine found (Docker/Finch/Podman not installed)
✅ Default deployment uses CodeBuild (no container engine needed)
💡 Run 'agentcore launch' for cloud-based building and deployment
💡 For local builds, install Docker, Finch, or Podman

⚠️  [WARNING] Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64'.
For deployment options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated .dockerignore
Generated Dockerfile: c:\allwyn\Dockerfile
Generated .dockerignore: c:\allwyn\.dockerignore
Setting 'agent' as default agent
Bedrock AgentCore configured: c:\allwyn\.bedrock_agentcore.yaml


ConfigureResult(config_path=WindowsPath('c:/allwyn/.bedrock_agentcore.yaml'), dockerfile_path=WindowsPath('c:/allwyn/Dockerfile'), dockerignore_path=WindowsPath('c:/allwyn/.dockerignore'), runtime='None', region='us-east-1', account_id='869795389844', execution_role='arn:aws:iam::869795389844:role/AgentCoreDemoRole', ecr_repository=None, auto_create_ecr=True)

In [14]:
launch_result = agent_runtime.launch(auto_update_on_conflict=True)

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'agent' to account 869795389844 (us-east-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: agent
✅ ECR repository available: 869795389844.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-agent
Using execution role from config: arn:aws:iam::869795389844:role/AgentCoreDemoRole
Preparing CodeBuild project and uploading source...


✅ Reusing existing ECR repository: 869795389844.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-agent


Getting or creating CodeBuild execution role for agent: agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-d4f0bc5a29
Reusing existing CodeBuild execution role: arn:aws:iam::869795389844:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-d4f0bc5a29
Using .dockerignore with 44 patterns
Uploaded source to S3: agent/source.zip
Updated CodeBuild project: bedrock-agentcore-agent-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.2s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.1s
🔄 DOWNLOAD_SOURCE started (total: 8s)
✅ DOWNLOAD_SOURCE completed in 1.2s
🔄 BUILD started (total: 10s)
✅ BUILD completed in 19.9s
🔄 POST_BUILD started (total: 30s)
✅ POST_BUILD completed in 14.1s
🔄 COMPLETED started (total: 44s)
✅ COMPLETED completed in 1.2s
🎉 CodeBuild completed successfully in 0m 44s
CodeBuild completed successfully
✅ CodeBuild project configuration saved
Deploying t

In [15]:
invoke_response = agent_runtime.invoke({"prompt": "How is the weather now in Athens ? Cross check with information from the web."})
invoke_response

{'ResponseMetadata': {'RequestId': 'd3ebf350-775a-4582-93ee-c445b82ca46b',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 24 Apr 2026 19:15:44 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'd3ebf350-775a-4582-93ee-c445b82ca46b',
   'x-amzn-bedrock-agentcore-runtime-session-id': 'e3187e9c-47a2-4bc0-9c49-d37bd58b095e'},
  'RetryAttempts': 0},
 'runtimeSessionId': 'e3187e9c-47a2-4bc0-9c49-d37bd58b095e',
 'contentType': 'application/json',
 'statusCode': 200,
 'response': ["Here's a cross-referenced summary of the current weather in **Athens, Greece**:\n\n| Source | Details |\n|--------|---------|\n| **Weather Tool** | Very sunny / clear skies |\n| **Web Search (WeatherAPI)** | **Clear**, 14.4°C (57.9°F), Humidity: 63%, Wind: NNE at 10.8 km/h (6.7 mph), Cloud cover: 25% |\n| **Weather Underground** | Fair, 64°F, Wind: N at 2 mph with gusts up to 7 mph |\n| **Meteobox** | Clear to partly cloudy

# MCP Server on  AgentCore
<hr>

In [16]:
mcp_server_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = mcp_server_runtime.configure(
    entrypoint="01_mcp_server.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    protocol="MCP",
    agent_name='mpc_server',
)

Entrypoint parsed: file=C:\allwyn\01_mcp_server.py, bedrock_agentcore_name=01_mcp_server
Configuring BedrockAgentCore agent: mpc_server


Configuring AgentCore Runtime...


⚠️  ℹ️  No container engine found (Docker/Finch/Podman not installed)
✅ Default deployment uses CodeBuild (no container engine needed)
💡 Run 'agentcore launch' for cloud-based building and deployment
💡 For local builds, install Docker, Finch, or Podman

⚠️  [WARNING] Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64'.
For deployment options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated Dockerfile: c:\allwyn\Dockerfile
Generated .dockerignore: c:\allwyn\.dockerignore
Changing default agent from 'agent' to 'mpc_server'
Bedrock AgentCore configured: c:\allwyn\.bedrock_agentcore.yaml


In [17]:
mcp_server_runtime.launch(auto_update_on_conflict=True)

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'mpc_server' to account 869795389844 (us-east-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: mpc_server
✅ ECR repository available: 869795389844.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-mpc_server
Using execution role from config: arn:aws:iam::869795389844:role/AgentCoreDemoRole
Preparing CodeBuild project and uploading source...


✅ Reusing existing ECR repository: 869795389844.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-mpc_server


Getting or creating CodeBuild execution role for agent: mpc_server
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-4daed537fa
Reusing existing CodeBuild execution role: arn:aws:iam::869795389844:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-4daed537fa
Using .dockerignore with 44 patterns
Uploaded source to S3: mpc_server/source.zip
Updated CodeBuild project: bedrock-agentcore-mpc_server-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.2s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.0s
🔄 DOWNLOAD_SOURCE started (total: 8s)
✅ DOWNLOAD_SOURCE completed in 1.2s
🔄 BUILD started (total: 10s)
✅ BUILD completed in 18.8s
🔄 POST_BUILD started (total: 28s)
✅ POST_BUILD completed in 14.1s
🔄 COMPLETED started (total: 42s)
✅ COMPLETED completed in 1.2s
🎉 CodeBuild completed successfully in 0m 43s
CodeBuild completed successfully
✅ CodeBuild project configuration sa

LaunchResult(mode='codebuild', tag='bedrock_agentcore-mpc_server:latest', env_vars=None, port=None, runtime=None, ecr_uri='869795389844.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-mpc_server', agent_id='mpc_server-IbPAU23Iwf', agent_arn='arn:aws:bedrock-agentcore:us-east-1:869795389844:runtime/mpc_server-IbPAU23Iwf', codebuild_id='bedrock-agentcore-mpc_server-builder:9e766902-131a-428f-9427-dab5a13cbd38', build_output=None)

# Agent -> MCP Server on  AgentCore
<hr>

In [18]:
agent_runtime = Runtime()

agent_runtime.configure(
    agent_name="agent",
    entrypoint="02_agent.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region
)

Entrypoint parsed: file=C:\allwyn\02_agent.py, bedrock_agentcore_name=02_agent
Configuring BedrockAgentCore agent: agent


⚠️  ℹ️  No container engine found (Docker/Finch/Podman not installed)
✅ Default deployment uses CodeBuild (no container engine needed)
💡 Run 'agentcore launch' for cloud-based building and deployment
💡 For local builds, install Docker, Finch, or Podman

⚠️  [WARNING] Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64'.
For deployment options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated Dockerfile: c:\allwyn\Dockerfile
Generated .dockerignore: c:\allwyn\.dockerignore
Changing default agent from 'mpc_server' to 'agent'
Bedrock AgentCore configured: c:\allwyn\.bedrock_agentcore.yaml


ConfigureResult(config_path=WindowsPath('c:/allwyn/.bedrock_agentcore.yaml'), dockerfile_path=WindowsPath('c:/allwyn/Dockerfile'), dockerignore_path=WindowsPath('c:/allwyn/.dockerignore'), runtime='None', region='us-east-1', account_id='869795389844', execution_role='arn:aws:iam::869795389844:role/AgentCoreDemoRole', ecr_repository=None, auto_create_ecr=True)

In [19]:
agent_runtime.launch(auto_update_on_conflict=True)

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'agent' to account 869795389844 (us-east-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: agent
✅ ECR repository available: 869795389844.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-agent
Using execution role from config: arn:aws:iam::869795389844:role/AgentCoreDemoRole
Preparing CodeBuild project and uploading source...


✅ Reusing existing ECR repository: 869795389844.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-agent


Getting or creating CodeBuild execution role for agent: agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-d4f0bc5a29
Reusing existing CodeBuild execution role: arn:aws:iam::869795389844:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-d4f0bc5a29
Using .dockerignore with 44 patterns
Uploaded source to S3: agent/source.zip
Updated CodeBuild project: bedrock-agentcore-agent-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.2s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.0s
🔄 DOWNLOAD_SOURCE started (total: 8s)
✅ DOWNLOAD_SOURCE completed in 1.2s
🔄 BUILD started (total: 9s)
✅ BUILD completed in 17.5s
🔄 POST_BUILD started (total: 27s)
✅ POST_BUILD completed in 14.1s
🔄 FINALIZING started (total: 41s)
✅ FINALIZING completed in 1.2s
🔄 COMPLETED started (total: 42s)
✅ COMPLETED completed in 1.2s
🎉 CodeBuild completed successfully in 0m 43s
CodeBuild completed 

LaunchResult(mode='codebuild', tag='bedrock_agentcore-agent:latest', env_vars=None, port=None, runtime=None, ecr_uri='869795389844.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-agent', agent_id='agent-Oi1i867z37', agent_arn='arn:aws:bedrock-agentcore:us-east-1:869795389844:runtime/agent-Oi1i867z37', codebuild_id='bedrock-agentcore-agent-builder:a3a11b04-47c9-4a7f-869a-ea36044adc3e', build_output=None)

In [22]:
response = agent_runtime.invoke({"prompt": "How is the weather now in Athens ? Cross check your answer with web information."})
response

{'ResponseMetadata': {'RequestId': 'bd6d39be-5d24-4a90-8584-729e5f7d988b',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 24 Apr 2026 19:22:26 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'bd6d39be-5d24-4a90-8584-729e5f7d988b',
   'x-amzn-bedrock-agentcore-runtime-session-id': 'f28f4226-5ded-4082-aef7-1f16ec1eceb0'},
  'RetryAttempts': 0},
 'runtimeSessionId': 'f28f4226-5ded-4082-aef7-1f16ec1eceb0',
 'contentType': 'application/json',
 'statusCode': 200,
 'response': ['"Here\'s the cross-checked weather report for **Athens, Greece** 🇬🇷:\\n\\n---\\n\\n### 🌙 Current Weather in Athens (as of April 24, 2026 — 10:20 PM local time)\\n\\n| Detail | Weather Tool | Web Sources |\\n|---|---|---|\\n| **Condition** | ☀️ Very Sunny | 🌙 **Clear skies** (WeatherAPI) |\\n| **Temperature** | — | **14.2°C / 57.6°F** (WeatherAPI) |\\n| **Feels Like** | — | **13.6°C / 56.4°F** |\\n| **Humidity** | — | **63%*

# MCP Gateway
<hr>

#### Create Gateway

In [23]:
agentcore_control_client = boto3.client("bedrock-agentcore-control")

# Create new gateway
response = agentcore_control_client.create_gateway(
    name='gateway',
    roleArn=role_arn,
    protocolType="MCP",
    authorizerType="AWS_IAM",
)

gateway_id = response["gatewayId"]
gateway_url = response["gatewayUrl"]

print(f"Gateway URL: {gateway_url}")

ConflictException: An error occurred (ConflictException) when calling the CreateGateway operation: A gateway with name 'gateway' already exists in this account

#### Create gateway MCP target

In [ ]:
MCP_SERVER_ARN = "arn:aws:bedrock-agentcore:us-east-1:869795389844:runtime/mpc_server-IbPAU23Iwf"
encoded_arn = MCP_SERVER_ARN.replace(":", "%3A").replace("/", "%2F")

mcp_endpoint = (
    f"https://bedrock-agentcore.{region}.amazonaws.com"
    f"/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
)

target = agentcore_control_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name='mcp-server',
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": mcp_endpoint,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "GATEWAY_IAM_ROLE",
            "credentialProvider": {
                "iamCredentialProvider": {
                    "service": "bedrock-agentcore",
                    "region": region,
                }
            },
        }
    ],
)

print(f"Target ID: {target['targetId']}")
print(f"Target status: {target['status']}")

#### Create gateway REST API target

In [ ]:
ACCOUNT_ID = "869795389844"
S3_OPENAPI_URI = f"s3://agentcore-rest-api-{ACCOUNT_ID}/openapi.json"
REST_API_KEY_PROVIDER_ARN = "arn:aws:bedrock-agentcore:us-east-1:869795389844:token-vault/default/apikeycredentialprovider/rest-api-key"

print(f"\nAdding REST API target...")
rest_target = agentcore.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="rest-api",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "s3": {
                    "uri": S3_OPENAPI_URI,
                    "bucketOwnerAccountId": ACCOUNT_ID,
                }
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": REST_API_KEY_PROVIDER_ARN,
                    "credentialLocation": "HEADER",
                    "credentialParameterName": "x-api-key",
                }
            },
        }
    ],
)

print(f"REST API Target ID: {rest_target['targetId']}")
print(f"REST API Target status: {rest_target['status']}")

# Agent -> MCP Gateway -> MCP on AgentCore
<hr>

In [24]:
agent_runtime = Runtime()

agent_runtime.configure(
    agent_name="agent",
    entrypoint="03_agent.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region
)

Entrypoint parsed: file=C:\allwyn\03_agent.py, bedrock_agentcore_name=03_agent
Configuring BedrockAgentCore agent: agent


⚠️  ℹ️  No container engine found (Docker/Finch/Podman not installed)
✅ Default deployment uses CodeBuild (no container engine needed)
💡 Run 'agentcore launch' for cloud-based building and deployment
💡 For local builds, install Docker, Finch, or Podman

⚠️  [WARNING] Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64'.
For deployment options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated Dockerfile: c:\allwyn\Dockerfile
Generated .dockerignore: c:\allwyn\.dockerignore
Keeping 'agent' as default agent
Bedrock AgentCore configured: c:\allwyn\.bedrock_agentcore.yaml


ConfigureResult(config_path=WindowsPath('c:/allwyn/.bedrock_agentcore.yaml'), dockerfile_path=WindowsPath('c:/allwyn/Dockerfile'), dockerignore_path=WindowsPath('c:/allwyn/.dockerignore'), runtime='None', region='us-east-1', account_id='869795389844', execution_role='arn:aws:iam::869795389844:role/AgentCoreDemoRole', ecr_repository=None, auto_create_ecr=True)

In [25]:
launch_result = agent_runtime.launch(auto_update_on_conflict=True)

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'agent' to account 869795389844 (us-east-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: agent
✅ ECR repository available: 869795389844.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-agent
Using execution role from config: arn:aws:iam::869795389844:role/AgentCoreDemoRole
Preparing CodeBuild project and uploading source...


✅ Reusing existing ECR repository: 869795389844.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-agent


Getting or creating CodeBuild execution role for agent: agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-d4f0bc5a29
Reusing existing CodeBuild execution role: arn:aws:iam::869795389844:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-d4f0bc5a29
Using .dockerignore with 44 patterns
Uploaded source to S3: agent/source.zip
Updated CodeBuild project: bedrock-agentcore-agent-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.2s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 8.2s
🔄 DOWNLOAD_SOURCE started (total: 10s)
✅ DOWNLOAD_SOURCE completed in 1.2s
🔄 BUILD started (total: 11s)
✅ BUILD completed in 17.7s
🔄 POST_BUILD started (total: 28s)
✅ POST_BUILD completed in 14.1s
🔄 FINALIZING started (total: 42s)
✅ FINALIZING completed in 1.2s
🔄 COMPLETED started (total: 44s)
✅ COMPLETED completed in 0.0s
🎉 CodeBuild completed successfully in 0m 43s
CodeBuild complete

In [26]:
import uuid

agent_core_client = boto3.client("bedrock-agentcore")
agent_arn = launch_result.agent_arn  # get ARN from the toolkit

session_id = str(uuid.uuid4())  # one session for the whole conversation

def chat(prompt, account_id="cust-003"):
    payload = json.dumps({"prompt": prompt, "accountId": account_id}).encode()
    response = agent_core_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=session_id,
        payload=payload,
        qualifier="DEFAULT",
    )
    content = []
    for chunk in response.get("response", []):
        content.append(chunk.decode("utf-8"))
    result = json.loads("".join(content))
    print(result)
    return result

In [1]:
chat("what bets have I played so far ?")

NameError: name 'chat' is not defined

In [28]:
chat("Find me the best value football bet right now — check the events, compare the odds, pick the one with the highest potential return, and present it to me.")

Here's a full breakdown of all football odds right now:

---

### ⚽ Football Odds Comparison

| Event | League | Home Win | Draw | Away Win |
|-------|--------|----------|------|----------|
| **Arsenal vs Chelsea** | Premier League | 2.10 | 3.40 | 3.20 |
| **Barcelona vs Real Madrid** | La Liga | 1.85 | 3.60 | 4.00 |
| **Bayern Munich vs Dortmund** | Bundesliga | 1.70 | 3.80 | **4.50** ⭐ |

---

### 🏆 Best Value Pick: **Dortmund to beat Bayern Munich (Away Win)**

| Detail | Value |
|--------|-------|
| **Event** | Bayern Munich vs Dortmund (Bundesliga) |
| **Selection** | Away Win (Dortmund) |
| **Odds** | **4.50** — the highest across all football markets |
| **Kickoff** | Apr 26, 2026 – 5:30 PM UTC |
| **Your Balance** | $148.00 |
| **If you bet $50** | Potential return: **$225.00** |
| **If you bet $100** | Potential return: **$450.00** |
| **If you go all-in ($148)** | Potential return: **$666.00** 🔥 |

At **4.50 odds**, a Dortmund away win offers the highest potential return per 

"Here's a full breakdown of all football odds right now:\n\n---\n\n### ⚽ Football Odds Comparison\n\n| Event | League | Home Win | Draw | Away Win |\n|-------|--------|----------|------|----------|\n| **Arsenal vs Chelsea** | Premier League | 2.10 | 3.40 | 3.20 |\n| **Barcelona vs Real Madrid** | La Liga | 1.85 | 3.60 | 4.00 |\n| **Bayern Munich vs Dortmund** | Bundesliga | 1.70 | 3.80 | **4.50** ⭐ |\n\n---\n\n### 🏆 Best Value Pick: **Dortmund to beat Bayern Munich (Away Win)**\n\n| Detail | Value |\n|--------|-------|\n| **Event** | Bayern Munich vs Dortmund (Bundesliga) |\n| **Selection** | Away Win (Dortmund) |\n| **Odds** | **4.50** — the highest across all football markets |\n| **Kickoff** | Apr 26, 2026 – 5:30 PM UTC |\n| **Your Balance** | $148.00 |\n| **If you bet $50** | Potential return: **$225.00** |\n| **If you bet $100** | Potential return: **$450.00** |\n| **If you go all-in ($148)** | Potential return: **$666.00** 🔥 |\n\nAt **4.50 odds**, a Dortmund away win offers the h

In [ ]:
chat("great! place it for 55$")

Your bet has been placed! 🎉

| Detail | Value |
|--------|-------|
| **Bet ID** | bet-c446167b |
| **Event** | Bayern Munich vs Dortmund |
| **Selection** | Away Win (Dortmund) |
| **Stake** | $55.00 |
| **Odds** | 4.50 |
| **Potential Return** | **$247.50** 💰 |
| **Remaining Balance** | $93.00 |

Good luck — let's go Dortmund! ⚽🤞 Let me know if you need anything else.


"Your bet has been placed! 🎉\n\n| Detail | Value |\n|--------|-------|\n| **Bet ID** | bet-c446167b |\n| **Event** | Bayern Munich vs Dortmund |\n| **Selection** | Away Win (Dortmund) |\n| **Stake** | $55.00 |\n| **Odds** | 4.50 |\n| **Potential Return** | **$247.50** 💰 |\n| **Remaining Balance** | $93.00 |\n\nGood luck — let's go Dortmund! ⚽🤞 Let me know if you need anything else."

# Clean Up